# Dataset and Dataloader 

Till now, we used batch gradient descent and the problem with it are as follows: 
- Problems:
1. memory inefficient
2. Better convergence

- Solution:
Using batches of data to train the model. (Mini batch gradient descent)

**Implementation of mini batch gradient descent**

Pytorch has two classes which are dataset and dataloader.

### Dataset and Dataloader classes

-  Dataset and Dataloader are core abstraction in pytorch that decouple how you define your data from how you efficiently iterate over it in training loops. 

1. Dataset class: <br>
    The dataset class is essentially blueprint. when we creare a custom dataset and decide how data is loaded and returned. 
    It defines:
    - `__init__()` which tells how data should be loaded.
    - `__len__()` which returns total number of samples
    - ` __getitem__(index)` which returns the data (and label) at the given index. 

<br>

2. Dataloader Class:  <br>
    The dataloader wraps a dataset and handles batching, shuffling and parallel loading for you. 
    Dataloader Control flow:
    - At the start of each epoch, the dataloader (if shuffle=True) shuffles indices (using a sampler).
    - It divies the indices into chuncks of batch_size.
    - for each index in chunk, data samples are fetched from the dataset object.
    - The samples are then collected and combined into a batch (using collate_fn).
    - The batch is returned to main training loop.

In [43]:
from sklearn.datasets import make_classification
import torch

In [44]:
# step 1

X,y =make_classification(
n_samples=10,   # Number of samples
n_features=2,   # number of features
n_informative=2,  # number of informative features
n_redundant=0, # number of redundant features
n_classes=2,   # Number of Classes
random_state=42   # For reproductibilty
)

X.shape

(10, 2)

In [45]:
# cobnvert the data to pytorch sensors

X=torch.tensor(X,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.long)
X

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388],
        [-2.8954,  1.9769],
        [-0.7206, -0.9606],
        [-1.9629, -0.9923],
        [-0.9382, -0.5430],
        [ 1.7273, -1.1858],
        [ 1.7774,  1.5116],
        [ 1.8997,  0.8344],
        [-0.5872, -1.9717]])

In [46]:
from torch.utils.data import Dataset, DataLoader

In [47]:
class CustomDataset(Dataset):

    def __init__(self,features,labels):
        self.features=features
        self.labels=labels

    
    def __len__(self):
        return self.features.shape[0]

    def __getitem__(self,index):
        return self.features[index],self.labels[index]

# object of dataset
dataset=CustomDataset(X,y)
len(dataset)

10

In [48]:
# dataloader class

dataloader=DataLoader(dataset,batch_size=2,shuffle=True)   #means in 0ne batch there will be two rows

In [49]:
for batch_features,batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-"*50)

tensor([[-2.8954,  1.9769],
        [-0.5872, -1.9717]])
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [ 1.7273, -1.1858]])
tensor([1, 1])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [ 1.0683, -0.9701]])
tensor([0, 1])
--------------------------------------------------
tensor([[-1.1402, -0.8388],
        [ 1.7774,  1.5116]])
tensor([0, 1])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [-0.7206, -0.9606]])
tensor([1, 0])
--------------------------------------------------


```TEXT
In dataloader, there is concept like worker, which will help in parallel processing. First chunck will be taken by first worker and parallely second worker will do same as well. so, Parellization is also possible.  

In Pytorch, the sampler in the dataloader determines the strategy for selecting samples from the dataset during data loading. It controls how indices of dataset are drawn for each batch.

Types of Samplers:
Pytorch provides several predefined samplers, and we can create custom ones:

1. Sequential samplers
    i. Samples elements sequentially, in order they appear in the dataset.
    ii. Default when shuffle=False

2. RandomSampler
    i. Samples elements randomly without replacement.
    ii. Default when shuffle=True

Use case of custom sample (sampler), in imbalance dataset. 


```


- Note about `collate_fn`:

        It is a function in dataloader that specifies how to combine a list of samples from dsataset into single batch. by default, the dataloader uses a sample batch collation mechanism, but collat_fn allows to customize how the data should be processed and batched.

    for example: logic of padding

### DataLoader Important Parameters

DataLoader comes with several Parameters that allow to customize how data is loaded, batched and preprocessed. some of the most commonly used and some are as follows:

1. dataset(mandatory)   
    - Dataset from which DataLoader will pull data
    - Must be a subclass of torch.utils.data.Dataset that implements ` __getitem__ and __len__.`

<br>

2. Batch_size:
    - how many samples per batch to load
    - Default is 1
    - Larger batch sizes can speed up training on GPUs but require more memory.

<br>

3. Shuffle:
    - if true, the DataLoader will shuffle the dataset indices each epoch.
    - helpful to avoid the model becoming too dependent on order of samples. 

<br>

4. num_workers:
    - The numbers of worker processes used to load data in parallel
    - seting num_workers>0 can speed up data loading by leveraging multiple CPU cores especially I/O or preprocessing is a bottleneck.


<br>

5. Pin_memory
    - If true, the dataloader will copy tensors into pinned (page-locked) memory before.
    - This can improve GPU Transfers speed and thus overall training throughput, particularly on CUDA systems.

<br>

6.  Drop_last:
    - If True, the Dataloader will drop the last incomplete batch, if total numvber of samples is not divisible by batch size.
    - useful when exact batch sizes are required ( for example, in some batch normalization scenarios)


<br>

7. collate_fn:
    - A callable that process a list of samples into a batch (the default simply stacks tensors).
    - custom collate_fn can handle variable length sequences, perform custom batching logic, or handle complex data structures.

<br>

9. Sampler:
    - Sampler defines the strategy for drawing samples (eg.. For handling imbalanced classes or custom sampling strategies);.
    - batch_sampler works at batch level, controlling how batches are formed.
    - typically, we dont need to specify these if you we are using batch_size and shuffle. However, they provide lower-level control if you have advanced requirements.

In [50]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


df=pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()
df.describe()

df.drop(columns=["id","Unnamed: 32"],inplace=True)
df.head(n=5)

X_train, X_test,y_train,y_test=train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)



scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

X_train_tensor=torch.from_numpy(X_train)
X_test_tensor=torch.from_numpy(X_test)
y_train_tensor=torch.from_numpy(y_train)
y_test_tensor=torch.from_numpy(y_test)

from torch.utils.data import Dataset,DataLoader

class CustomDataset(Dataset):

    def __init__(self,features,labels):
        self.features=features
        self.labels=labels

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):
        return self.features[idx],self.labels[idx]


In [51]:
train_dataset=CustomDataset(X_train_tensor,y_train_tensor)
test_dataset=CustomDataset(X_test_tensor,y_test_tensor)
train_dataset[10]

(tensor([-3.7258e-01, -7.9652e-01, -3.6282e-01, -4.2464e-01, -7.0805e-01,
         -1.9013e-01, -5.0709e-01, -4.8253e-01, -1.1650e-01, -5.0806e-02,
         -6.8513e-01, -8.3769e-01, -5.7816e-01, -5.1395e-01, -5.6959e-01,
         -5.4704e-01, -5.6072e-01, -6.9726e-01,  6.1299e-02, -7.2313e-01,
         -3.9336e-01, -6.4045e-01, -3.2245e-01, -4.3373e-01, -6.4860e-02,
          3.9053e-04, -3.0176e-01, -2.1578e-01,  1.0096e+00, -1.4014e-01],
        dtype=torch.float64),
 tensor(0))

In [52]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [53]:
import torch.nn as nn

class MySimpleNN(nn.Module):

    def __init__(self,num_features):
        super().__init__()
        self.linear=nn.Linear(num_features,1)
        self.sigmoid=nn.Sigmoid()

    def forward(self,features):
        out=self.linear(features)
        out=self.sigmoid(out)

        return out
    

In [54]:
learning_rate=0.1
epochs=25

model=MySimpleNN(X_train_tensor.shape[1])
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
loss_function=nn.BCELoss()

In [62]:
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:

        batch_features = batch_features.float()
        batch_labels = batch_labels.float()

        y_pred = model(batch_features)

        loss = loss_function(y_pred, batch_labels.view(-1,1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    print(f"Epoch:{epoch+1},loss:{loss.item()}")

Epoch:1,loss:0.004116324242204428
Epoch:2,loss:0.0019720965065062046
Epoch:3,loss:0.20180046558380127
Epoch:4,loss:0.011227303184568882
Epoch:5,loss:0.03720502927899361
Epoch:6,loss:0.004953671712428331
Epoch:7,loss:0.04921787232160568
Epoch:8,loss:0.004715602844953537
Epoch:9,loss:0.003749482799321413
Epoch:10,loss:0.04911785572767258
Epoch:11,loss:0.02896127477288246
Epoch:12,loss:0.09257109463214874
Epoch:13,loss:0.003353958949446678
Epoch:14,loss:0.08974404633045197
Epoch:15,loss:0.09814702719449997
Epoch:16,loss:0.003736890619620681
Epoch:17,loss:0.19123007357120514
Epoch:18,loss:0.020848551765084267
Epoch:19,loss:0.009763539768755436
Epoch:20,loss:0.03756704926490784
Epoch:21,loss:0.15044744312763214
Epoch:22,loss:0.017901061102747917
Epoch:23,loss:0.009887805208563805
Epoch:24,loss:0.0067000724375247955
Epoch:25,loss:0.05845721438527107


In [65]:
model.eval()
accuracy_list=[]

with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        # forward pass

        batch_features = batch_features.float()
        batch_labels = batch_labels.float()

        y_pred=model(batch_features).float()
        y_pred=(y_pred>0.8).float()

        batch_accuracy=(y_pred.view(-1)==batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

overall_accuracy=sum(accuracy_list)/len(accuracy_list)
print(f"Accuracy: {overall_accuracy:.4f}")

Accuracy: 0.9705
